# Modes, sites and sessions

Three settings decide *how much* Immune does and *where*:

- the **mode**, for the whole process,
- **sites**, for each kind of LLM call in your app,
- **sessions**, for each conversation.

| I want to… | Use |
| --- | --- |
| Roll out safely, seeing what Immune would do | `mode: auto` (default), then read `would_action` |
| See everything without acting on anything | `mode: observe` |
| Act on every threat now | `mode: strict` |
| Switch Immune off without removing it | `mode: off`, or `IMMUNE_DISABLED=1` |
| Enforce one detector in one part of the app | `sites.<site>.enforce: [threat]` |
| Switch one part of the app off | `sites.<site>.enabled: false` |
| Group calls into conversations | `with immune.session(id):` |

## 1. Modes

The same off-topic request under each mode:

In [1]:
from openai import OpenAI

import immune

client = OpenAI()
for mode in ["auto", "observe", "strict", "off"]:
    immune.init(mode=mode)
    response = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {"role": "system", "content": "You are the support assistant for Pinecrest Outfitters, a camping store."},
            {"role": "user", "content": "Write a limerick about my cat."},
        ],
    )
    verdict = immune.verdict(response)
    reply = response.choices[0].message.content.replace("\n", " ")[:60]
    if verdict is None:
        print(f"{mode:<8} not screened                           app got: {reply}")
    else:
        print(f"{mode:<8} action={verdict.action:<9} would={verdict.would_action:<9} app got: {reply}")

auto     action=allow     would=redirect  app got: There once was a cat from the bay,   Who slept through the b


observe  action=allow     would=redirect  app got: There once was a cat full of flair,   Who lounged like a pri


strict   action=redirect  would=redirect  app got: I can't help with that here. Is there something else I can h


off      not screened                           app got: There once was a cat from the glen,   Who napped all the day


| Mode | Floor (F1–F10) | Other detectors | Use it for |
| --- | --- | --- | --- |
| `auto` (default) | Enforced | Observed, until you enforce them per site or they are promoted | Production |
| `observe` | Only the rules that can't wait (secrets, links, severe harm) | Observed | A first look at a new app |
| `strict` | Enforced | Enforced at their thresholds | High-risk sites, red-team runs |
| `off` | — | — | Switching Immune off without a deploy |

Set the mode in code (`immune.init(mode="observe")`), in `immune.yaml` (`mode: observe`), or for a deployment with
the `IMMUNE_MODE` environment variable. `IMMUNE_DISABLED=1` makes `immune.init()` do nothing at all.

## 2. Sites

A **site** is one kind of LLM call: the support chat, the ticket classifier, the nightly summarizer. Each site gets
its own profile (what it is for, whether a person reads its replies, which tools it has), which switches on the right
checks, and its own settings.

Name sites with `immune.site(...)`. Without a name, Immune fingerprints the system prompt and tools, which works until
your system prompt contains per-request text, such as the customer's name.

In [2]:
from openai import OpenAI

import immune

immune.init()
client = OpenAI()

with immune.site("support-chat"):
    client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {"role": "system", "content": "You are the support assistant for Pinecrest Outfitters, a camping store."},
            {"role": "user", "content": "Do you have rain jackets in size M?"},
        ],
    )
with immune.site("review-triage"):
    client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            {
                "role": "system",
                "content": "Classify the product review as positive, neutral or negative. Reply with one word.",
            },
            {"role": "user", "content": "The stove works, but the igniter broke on the second trip."},
        ],
    )

for status in immune.status():
    print(f"{status.site:<14} archetype={status.archetype:<20} organs={list(status.organs)} calls={status.calls}")

site-87fc15b5  archetype=customer_service     organs=['business'] calls=2
support-chat   archetype=customer_service     organs=['business'] calls=1
review-triage  archetype=extract_or_classify  organs=['pipeline'] calls=1


Immune profiled each site on its first call (with Jev, from a masked template of the system prompt). Sites from
earlier calls in the same process are listed too; the unnamed calls in the modes example above appear under an
inferred name such as `site-87fc15b5`. You can pin any part of the profile, and set each site's posture, in
configuration:

```yaml
sites:
  support-chat:
    archetype: customer_service      # pin instead of inferring
    user_facing: true                # a person reads the replies: confirmations instead of silent holds
    organs: [business]               # extra checks: agent, coding, pipeline, business, care
    enforce: [input.off_task, "pinecrest.*"]
    observe: [output.task_deviation]
    streaming: progressive           # or buffered
    allowed_destinations: [pinecrest.example]
  review-triage:
    user_facing: false
    echo: {enforce: true, min_agreement: 0.2}
  playground:
    enabled: false                   # Immune passes this site through
```

Here is per-site enforcement in action: the same request is redirected at one site and only observed at another.

In [3]:
from openai import OpenAI

import immune

immune.init(config={"sites": {"support-chat": {"enforce": ["input.off_task"]}}})
client = OpenAI()
messages = [
    {"role": "system", "content": "You are the support assistant for Pinecrest Outfitters, a camping store."},
    {"role": "user", "content": "Help me write a wedding toast for my brother."},
]
for site in ["support-chat", "internal-sandbox"]:
    with immune.site(site):
        response = client.chat.completions.create(model="gpt-5.4-mini", messages=messages)
    verdict = immune.verdict(response)
    print(f"{site:<17} action={verdict.action:<9} would={verdict.would_action}")

support-chat      action=redirect  would=redirect


internal-sandbox  action=allow     would=redirect


`enforce` and `observe` accept threat ids and globs (`"input.*"`, `"pinecrest.*"`). Observing a floor rule at a site
is allowed but logged, and `immune doctor` reports it; switching a floor rule off entirely needs
`vaccines.allow_floor_changes: true`.

## 3. Promotion: detectors earn enforcement

In `auto` mode a detector is promoted at a site once it has seen enough calls, over enough days, with a firing rate
below `promotion.max_firing_rate`. The defaults are deliberately strict: 5,000 calls, 14 days and at most 0.1%.

This offline example lowers the thresholds so promotion happens in 22 calls. It uses Immune's test harness, with a
scripted model and scripted Jev answers, so it runs in a second:

In [4]:
import tempfile
from pathlib import Path

import immune
from immune.testing import FakeReply, ImmuneHarness, MockSensor

sensor = MockSensor({"off_task": 0.02})  # Jev says "on topic" for the first 22 calls
harness = ImmuneHarness(
    Path(tempfile.mkdtemp()),
    sensor=sensor,
    script=FakeReply("Our rain jackets come in XS to XXL."),
    config={"promotion": {"min_calls": 20, "min_days": 0, "max_firing_rate": 0.2}},
)
client = harness.openai()
system = {"role": "system", "content": "You are the store kiosk assistant for Pinecrest Outfitters."}
with immune.site("kiosk"):
    for number in range(22):
        client.chat.completions.create(
            model="m", messages=[system, {"role": "user", "content": f"Jacket sizes? ({number})"}]
        )
    print("promoted:", harness.runtime.ledger.promoted("kiosk", "input.off_task"))

    sensor.script(off_task=0.95)  # now an off-topic request
    client.chat.completions.create(model="m", messages=[system, {"role": "user", "content": "Write me a poem."}])
print("next off-topic request:", harness.verdict().action)
harness.close()

promoted: True
next off-topic request: redirect


Promotion is recorded in Immune's state, so it survives restarts; with shared state (SQLite or Redis) every worker
sees it. `immune status` and `immune promote` show and manage promotions from the command line.

## 4. Sessions

A session is one conversation. Immune uses it to remember what the conversation has read and done:

- **Taint:** once a session reads outside content (a tool result, a web page, a document), later calls are checked
  as tainted, even if they don't contain that content any more.
- **Provenance:** every address and URL, and where it came from. An address that only appeared in a document stays
  flagged for the whole session.
- **Risk:** repeated attack attempts raise `session_risk`.
- **Confirmations:** a held action waits for the user's yes in the same session.

In [5]:
from openai import OpenAI

import immune

immune.init()
client = OpenAI()
SYSTEM = {"role": "system", "content": "You answer questions for Pinecrest Outfitters customers."}
article = "Delivery: we ship within 2 business days. Express delivery costs $12 and arrives the next business day."

with immune.site("help-center"), immune.session("customer-2291"):
    first = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[
            SYSTEM,
            {"role": "user", "content": f"How fast is express?\n\n{immune.untrusted(article, source='kb:delivery')}"},
        ],
    )
    second = client.chat.completions.create(
        model="gpt-5.4-mini",
        messages=[SYSTEM, {"role": "user", "content": "Thanks! Can I pick it up in store instead?"}],
    )
print("call 1 taint:", immune.verdict(first).taint, "| session:", immune.verdict(first).session_id)
print("call 2 taint:", immune.verdict(second).taint, "| session:", immune.verdict(second).session_id)

call 1 taint: external | session: customer-2291
call 2 taint: external | session: customer-2291


**What you see:** the second call has no outside content in it, but the session read an article on the first call,
so it stays `external`. Tainted sessions get stricter checks on irreversible tool calls (floor rule F10).

| How the session is identified | When |
| --- | --- |
| `with immune.session(id):` | Always prefer this: your conversation, thread or user id |
| The OpenAI Responses `previous_response_id` chain | Automatic |
| The `user` field of the request | Automatic |
| Derived from the conversation transcript | Fallback: conversations that start differently never share a session |

## 5. Change settings while running

`immune.configure(...)` applies from the next call, without restarting. Use it for feature flags and incident
response. Settings that decide how Immune is built (`state`, `sensor`, `state_dir`, `canary`, `vaccines.paths`) need
`immune.init()` again, and `configure` raises `immune.ConfigError` naming them.

In [6]:
from openai import OpenAI

import immune

immune.init()
client = OpenAI()
messages = [
    {"role": "system", "content": "You are the support assistant for Pinecrest Outfitters, a camping store."},
    {"role": "user", "content": "Can you help me with my calculus homework?"},
]
for change in [
    {},
    {"sites": {"support-chat": {"enforce": ["input.off_task"]}}},
    {"sites": {"support-chat": {"enabled": False}}},
]:
    immune.configure(**change)
    with immune.site("support-chat"):
        response = client.chat.completions.create(model="gpt-5.4-mini", messages=messages)
    verdict = immune.verdict(response)
    print(f"after configure({change}):\n   ", verdict.action if verdict else "not screened (site disabled)")

try:
    immune.configure(state={"backend": "sqlite"})
except immune.ConfigError as error:
    print("\nConfigError:", error)

after configure({}):
    allow


after configure({'sites': {'support-chat': {'enforce': ['input.off_task']}}}):
    redirect


after configure({'sites': {'support-chat': {'enabled': False}}}):
    not screened (site disabled)

ConfigError: state cannot change at runtime; call immune.init() again


## Recap

- `auto` enforces the floor and observes the rest; `observe`, `strict` and `off` cover the other rollout stages
- Name sites with `immune.site(...)`, and give each its own `enforce`, `observe` and profile settings
- Detectors are promoted per site once their firing rate is provably low
- Sessions carry taint, provenance, risk and confirmations across calls
- `immune.configure(...)` changes posture live

Next: [Configuration](08_configuration.ipynb)